In [3]:
# Extract all URLs using Python; first, place solar_articles.jsonl in the newsroom directory:

import json
from pathlib import Path

print("defining paths")

input_path = Path("newsroom/solar_articles.jsonl")
output_path = Path("newsroom/solar_urls.txt")

print(f"input path: {input_path}")
print(f"output path: {output_path}")

urls = []
missing = 0

with input_path.open(encoding="utf-8") as source:
    for line_number, line in enumerate(source, 1):
        if not line.strip():
            continue

        record = json.loads(line)
        archive_url = record.get("archive")

        if archive_url:
            urls.append(archive_url)
        else:
            missing += 1
            print(f"Line {line_number}: no archive field")

unique_urls = list(dict.fromkeys(urls))

with output_path.open("w", encoding="utf-8") as destination:
    destination.writelines(f"{url}\n" for url in unique_urls)

print(f"Wrote {len(unique_urls)} URLs to {output_path}")
print(f"Records missing archive URLs: {missing}")



defining paths
input path: newsroom/solar_articles.jsonl
output path: newsroom/solar_urls.txt
Wrote 5402 URLs to newsroom/solar_urls.txt
Records missing archive URLs: 0


Then, to scrape the archived pages, run:

newsroom-scrape \
  --urls solar_urls.txt \
  --archive solar_articles.archive \
  --workers 4 \
  --sleep 1 \
  --tries 5

Install a missing spaCy model (I have to add --break-system-packages to get around brew):

python3 -m spacy download en_core_web_sm --break-system-packages

Extract article text:

newsroom-extract \
--archive solar_articles.archive \
--dataset solar_articles_full.jsonl.gz \
--workers 4



In [2]:
# Verify extraction using Python:

import gzip
import json

path = "newsroom/solar_articles_full.jsonl.gz"

count = 0
first = None

with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        count += 1
        if first is None:
            first = record

print("Records:", count)
print("Fields:", sorted(first.keys()) if first else [])



Records: 5386
Fields: ['archive', 'compression', 'compression_bin', 'coverage', 'coverage_bin', 'date', 'density', 'density_bin', 'summary', 'text', 'title', 'url']


In [4]:
# Check which records have usable article text using Python:

import gzip
import json

path = "newsroom/solar_articles_full.jsonl.gz"

total = 0
with_text = 0
empty = []

with gzip.open(path, "rt", encoding="utf-8") as f:
    for line_number, line in enumerate(f, 1):
        record = json.loads(line)
        total += 1

        text = record.get("text") or ""
        if text.strip():
            with_text += 1
        else:
            empty.append({
                "line": line_number,
                "title": record.get("title"),
                "archive": record.get("archive"),
            })

print("Total:", total)
print("With text:", with_text)
print("Missing text:", len(empty))

for item in empty[:20]:
    print(item)

Total: 5386
With text: 5374
Missing text: 12
{'line': 679, 'title': 'Cheap Garden Upgrades', 'archive': 'http://web.archive.org/web/20160730012208id_/http://time.com:80/money/4346867/cheap-garden-tips/'}
{'line': 1385, 'title': '10 Travel Spots to Stay Off the Grid', 'archive': 'http://web.archive.org/web/20160817121131id_/http://time.com:80/money/4364142/4th-of-july-weekend-travel/?'}
{'line': 1891, 'title': 'Cost of Solar Panels: Buying, Renting', 'archive': 'http://web.archive.org/web/20160730012057id_/http://time.com:80/money/4299588/solar-panels-cost/?'}
{'line': 1896, 'title': 'The New York Times > Log In', 'archive': 'https://web.archive.org/web/2007080319id_/http://www.nytimes.com/2007/08/03/washington/03energy.html'}
{'line': 2989, 'title': 'Cookies are Not Accepted', 'archive': 'https://web.archive.org/web/2011062819id_/http://www.nytimes.com/2011/06/28/arts/design/steven-holls-design-for-the-vanke-center-in-china-review.html'}
{'line': 3128, 'title': 'Log In - New York Times

Before converting to CSV, install ftfy to fix encoding issues:

python3 -m pip install ftfy --break-system-packages

In [6]:
# Convert to a CSV file using Python:

import csv
import gzip
import html
import json
import re
import unicodedata
from pathlib import Path

from ftfy import fix_text


INPUT_PATH = Path("newsroom/solar_articles_full.jsonl.gz")
OUTPUT_PATH = Path("data/solar_articles.csv")


def clean_text(value, preserve_newlines=False):
    if value is None:
        return ""

    text = str(value)

    # Decode strings such as &amp;, &#39;, and &nbsp;.
    text = html.unescape(text)

    # Repair common UTF-8/Windows-1252 mojibake.
    text = fix_text(text)

    # Normalize equivalent Unicode representations.
    text = unicodedata.normalize("NFC", text)

    # Replace special spaces with ordinary spaces.
    text = (
        text.replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("\ufeff", "")
    )

    # Remove null bytes and non-printing control characters.
    text = text.replace("\x00", "")
    text = "".join(
        character
        for character in text
        if character in "\n\t" or unicodedata.category(character) != "Cc"
    )

    if preserve_newlines:
        # Clean spaces around lines while retaining paragraphs.
        lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
        text = "\n".join(lines)
        text = re.sub(r"\n{3,}", "\n\n", text)
    else:
        text = re.sub(r"\s+", " ", text).strip()

    return text.strip()


fields = [
    "title",
    "date",
    "url",
    "archive",
    "summary",
    "text",
    "original_text",
]

total = 0
written = 0
changed = 0
empty = 0

with gzip.open(INPUT_PATH, "rt", encoding="utf-8") as source, \
     OUTPUT_PATH.open("w", encoding="utf-8-sig", newline="") as destination:

    writer = csv.DictWriter(
        destination,
        fieldnames=fields,
        quoting=csv.QUOTE_MINIMAL,
    )
    writer.writeheader()

    for line_number, line in enumerate(source, 1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            print(f"Skipping invalid JSON on line {line_number}: {error}")
            continue

        total += 1

        original_text = record.get("text") or ""
        cleaned_text = clean_text(original_text, preserve_newlines=True)

        if not cleaned_text:
            empty += 1
            continue

        if cleaned_text != original_text:
            changed += 1

        writer.writerow({
            "title": clean_text(record.get("title")),
            "date": record.get("date", ""),
            "url": record.get("url", ""),
            "archive": record.get("archive", ""),
            "summary": clean_text(
                record.get("summary"),
                preserve_newlines=True,
            ),
            "text": cleaned_text,
            "original_text": original_text,
        })

        written += 1

print(f"Input records: {total}")
print(f"CSV rows written: {written}")
print(f"Text fields changed: {changed}")
print(f"Empty-text records skipped: {empty}")
print(f"Output: {OUTPUT_PATH}")

Input records: 5386
CSV rows written: 5374
Text fields changed: 3388
Empty-text records skipped: 12
Output: data/solar_articles.csv
